In [1]:
# ============================================================
# 05 - SUPER DATASET
# ============================================================

import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data.load_master_data import load_master_data

master = load_master_data()
print(f"Master shape: {master.shape}")

Master shape: (25939, 46)


In [2]:
# ============================================================
# LAST INN RÅFILER
# ============================================================

new_pit_stops = pd.read_csv("../data/raw/new_pit_stops.csv")
new_lap_times = pd.read_csv("../data/raw/new_lap_times.csv")
new_races     = pd.read_csv("../data/raw/new_races.csv")
f1_races      = pd.read_csv("../data/raw/f1_races.csv")
new_drivers   = pd.read_csv("../data/raw/new_drivers.csv")

print("new_pit_stops:", new_pit_stops.shape)
print("new_lap_times:", new_lap_times.shape)
print("new_races:", new_races.shape)
print("f1_races:", f1_races.shape)
print("new_drivers:", new_drivers.shape)

new_pit_stops: (11371, 7)
new_lap_times: (589081, 6)
new_races: (1125, 18)
f1_races: (1171, 7)
new_drivers: (861, 9)


In [3]:
# ============================================================
# BYGG MAPPINGER
# ============================================================

# Race mapping: raceId -> season + round (via date)
race_id_to_date = new_races[["raceId", "date"]].copy()
date_to_season_round = f1_races[["date", "season", "round"]].copy()

race_id_map = race_id_to_date.merge(
    date_to_season_round,
    on="date",
    how="inner"
)

print(f"Race mapping shape: {race_id_map.shape}")
print(race_id_map.head())

# Driver mapping: driverId -> driver_id
driver_id_map = new_drivers[["driverId", "driverRef"]].copy()
driver_id_map = driver_id_map.rename(columns={
    "driverRef": "driver_id"
})

print(f"\nDriver mapping shape: {driver_id_map.shape}")
print(driver_id_map.head())

Race mapping shape: (1125, 4)
   raceId        date  season  round
0       1  2009-03-29    2009      1
1       2  2009-04-05    2009      2
2       3  2009-04-19    2009      3
3       4  2009-04-26    2009      4
4       5  2009-05-10    2009      5

Driver mapping shape: (861, 2)
   driverId   driver_id
0         1    hamilton
1         2    heidfeld
2         3     rosberg
3         4      alonso
4         5  kovalainen


In [4]:
# ============================================================
# RYDD PIT_STOPS
# ============================================================

pit_clean = new_pit_stops.copy()

# Merge inn season + round
pit_clean = pit_clean.merge(
    race_id_map[["raceId", "season", "round"]],
    on="raceId",
    how="left"
)

# Merge inn driver_id
pit_clean = pit_clean.merge(
    driver_id_map,
    on="driverId",
    how="left"
)

print(f"NaN i season: {pit_clean['season'].isnull().sum()}")
print(f"NaN i round: {pit_clean['round'].isnull().sum()}")
print(f"NaN i driver_id: {pit_clean['driver_id'].isnull().sum()}")

NaN i season: 0
NaN i round: 0
NaN i driver_id: 0


In [5]:
# Konverter duration til sekunder
def pit_duration_to_seconds(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    try:
        if ":" in val:
            parts = val.split(":")
            return float(parts[0]) * 60 + float(parts[1])
        else:
            return float(val)
    except:
        return np.nan

pit_clean["duration_seconds"] = (
    pit_clean["duration"].apply(pit_duration_to_seconds)
)

# Fjern urealistiske verdier
pit_clean = pit_clean[
    pit_clean["duration_seconds"].between(2, 120)
    | pit_clean["duration_seconds"].isna()
]

# Fjern unødvendige kolonner
pit_clean = pit_clean.drop(columns=[
    "raceId", "driverId", "time", "duration", "milliseconds"
])

print(pit_clean.columns.tolist())
print(f"Shape: {pit_clean.shape}")

['stop', 'lap', 'season', 'round', 'driver_id', 'duration_seconds']
Shape: (10890, 6)


In [6]:
# Aggreger til én rad per sjåfør per løp
# Dette er faktiske verdier per løp - brukes til å lage historikk
pit_per_race = pit_clean.groupby(
    ["season", "round", "driver_id"]
).agg(
    total_pit_stops  = ("stop", "count"),
    avg_pit_duration = ("duration_seconds", "mean"),
    min_pit_duration = ("duration_seconds", "min"),
).reset_index()

print(f"Shape: {pit_per_race.shape}")
print(pit_per_race.head(10))

Shape: (5534, 6)
   season  round    driver_id  total_pit_stops  avg_pit_duration  \
0    2011      1  alguersuari                3         25.903000   
1    2011      1       alonso                3         24.055000   
2    2011      1     ambrosio                2         25.852500   
3    2011      1  barrichello                4         26.175000   
4    2011      1        buemi                2         24.221000   
5    2011      1       button                3         20.950333   
6    2011      1        glock                1         23.792000   
7    2011      1     hamilton                2         23.213000   
8    2011      1     heidfeld                2         24.046000   
9    2011      1    kobayashi                2         24.128000   

   min_pit_duration  
0            24.463  
1            23.251  
2            25.259  
3            16.892  
4            23.100  
5            16.867  
6            23.792  
7            23.199  
8            22.994  
9            2

In [7]:
# ============================================================
# LAG HISTORISKE PIT FEATURES
# Bruker kun data fra TIDLIGERE løp - ingen leakage
# ============================================================

# Sorter på sesong og runde for å sikre riktig rekkefølge
pit_per_race = pit_per_race.sort_values(
    ["driver_id", "season", "round"]
).reset_index(drop=True)

# Historisk snitt antall pitstop per sjåfør
pit_per_race["hist_avg_pit_stops"] = (
    pit_per_race.groupby("driver_id")["total_pit_stops"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Historisk snitt pitstop-varighet per sjåfør
pit_per_race["hist_avg_pit_duration"] = (
    pit_per_race.groupby("driver_id")["avg_pit_duration"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Historisk beste pitstop per sjåfør
pit_per_race["hist_best_pit_duration"] = (
    pit_per_race.groupby("driver_id")["min_pit_duration"]
    .transform(lambda x: x.shift(1).expanding().min())
)

# Behold kun nøkler og historiske features - ikke leakage-kolonner
pit_historical = pit_per_race[[
    "season", "round", "driver_id",
    "hist_avg_pit_stops",
    "hist_avg_pit_duration",
    "hist_best_pit_duration"
]].copy()

print(pit_historical.head(20))
print(f"\nNaN i historiske features:")
print(pit_historical.isnull().sum())

    season  round driver_id  hist_avg_pit_stops  hist_avg_pit_duration  \
0     2020     16    aitken                 NaN                    NaN   
1     2019      1     albon                 NaN                    NaN   
2     2019      2     albon            1.000000              21.780000   
3     2019      3     albon            1.500000              23.515250   
4     2019      4     albon            1.333333              23.415167   
5     2019      5     albon            1.250000              22.741375   
6     2019      6     albon            1.400000              24.286700   
7     2019      7     albon            1.333333              24.415917   
8     2019      8     albon            1.285714              26.349357   
9     2019      9     albon            1.250000              26.840563   
10    2019     10     albon            1.222222              26.216611   
11    2019     11     albon            1.200000              26.490350   
12    2019     12     albon           

In [8]:
# ============================================================
# RYDD LAP_TIMES
# ============================================================

lap_clean = new_lap_times.copy()

# Merge inn season + round
lap_clean = lap_clean.merge(
    race_id_map[["raceId", "season", "round"]],
    on="raceId",
    how="left"
)

# Merge inn driver_id
lap_clean = lap_clean.merge(
    driver_id_map,
    on="driverId",
    how="left"
)

print(f"NaN i season: {lap_clean['season'].isnull().sum()}")
print(f"NaN i round: {lap_clean['round'].isnull().sum()}")
print(f"NaN i driver_id: {lap_clean['driver_id'].isnull().sum()}")

NaN i season: 0
NaN i round: 0
NaN i driver_id: 0


In [9]:
# Konverter milliseconds til sekunder
lap_clean["lap_time_seconds"] = lap_clean["milliseconds"] / 1000

# Fjern urealistiske verdier
lap_clean = lap_clean[
    lap_clean["lap_time_seconds"].between(50, 300)
    | lap_clean["lap_time_seconds"].isna()
]

# Fjern unødvendige kolonner
lap_clean = lap_clean.drop(columns=[
    "raceId", "driverId", "time", "milliseconds"
])

print(lap_clean.columns.tolist())
print(f"Shape: {lap_clean.shape}")

['lap', 'position', 'season', 'round', 'driver_id', 'lap_time_seconds']
Shape: (588373, 6)


In [10]:
# Aggreger til én rad per sjåfør per løp
lap_per_race = lap_clean.groupby(
    ["season", "round", "driver_id"]
).agg(
    avg_lap_time = ("lap_time_seconds", "mean"),
    min_lap_time = ("lap_time_seconds", "min"),
    lap_time_std = ("lap_time_seconds", "std"),
).reset_index()

print(f"Shape: {lap_per_race.shape}")
print(lap_per_race.head(10))

Shape: (11040, 6)
   season  round    driver_id  avg_lap_time  min_lap_time  lap_time_std
0    1996      1        alesi     97.365889        95.519      3.488146
1    1996      1  barrichello     98.117655        95.064      5.386448
2    1996      1       berger     97.371172        94.757      4.663494
3    1996      1      brundle    116.481000       116.481           NaN
4    1996      1    coulthard     99.797708        97.746      3.823771
5    1996      1   damon_hill     96.042948        93.621      4.813281
6    1996      1        diniz    100.101589        97.024      6.509712
7    1996      1   fisichella    101.550656        98.077      8.000509
8    1996      1     frentzen     98.411158        95.596      6.124687
9    1996      1     hakkinen     97.682103        95.843      4.255012


In [11]:
# ============================================================
# LAG HISTORISKE LAP FEATURES
# Bruker kun data fra TIDLIGERE løp - ingen leakage
# ============================================================

# Sorter på sesong og runde
lap_per_race = lap_per_race.sort_values(
    ["driver_id", "season", "round"]
).reset_index(drop=True)

# Historisk snitt-rundetid per sjåfør
lap_per_race["hist_avg_lap_time"] = (
    lap_per_race.groupby("driver_id")["avg_lap_time"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Historisk beste rundetid per sjåfør
lap_per_race["hist_best_lap_time"] = (
    lap_per_race.groupby("driver_id")["min_lap_time"]
    .transform(lambda x: x.shift(1).expanding().min())
)

# Historisk konsistens per sjåfør (lavere = mer konsistent)
lap_per_race["hist_lap_consistency"] = (
    lap_per_race.groupby("driver_id")["lap_time_std"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Historisk snitt per sjåfør per bane
# For dette trenger vi circuit_id - merge inn fra master
circuit_map = master[["season", "round", "circuit_id"]].drop_duplicates()

lap_per_race = lap_per_race.merge(
    circuit_map,
    on=["season", "round"],
    how="left"
)

# Historisk beste rundetid per sjåfør PER BANE
lap_per_race["hist_best_lap_at_circuit"] = (
    lap_per_race.groupby(["driver_id", "circuit_id"])["min_lap_time"]
    .transform(lambda x: x.shift(1).expanding().min())
)

# Behold kun nøkler og historiske features
lap_historical = lap_per_race[[
    "season", "round", "driver_id",
    "hist_avg_lap_time",
    "hist_best_lap_time",
    "hist_lap_consistency",
    "hist_best_lap_at_circuit"
]].copy()

print(lap_historical.head(20))
print(f"\nNaN i historiske features:")
print(lap_historical.isnull().sum())

    season  round driver_id  hist_avg_lap_time  hist_best_lap_time  \
0     2020     16    aitken                NaN                 NaN   
1     2005      1    albers                NaN                 NaN   
2     2005      2    albers          95.856562              93.144   
3     2005      3    albers         101.079204              93.144   
4     2005      4    albers         101.098199              93.144   
5     2005      5    albers          98.182249              87.420   
6     2005      6    albers          94.871688              80.124   
7     2005      7    albers          93.626087              80.124   
8     2005      8    albers          94.284205              80.124   
9     2005      9    albers          92.847021              78.462   
10    2005     10    albers          91.025692              73.907   
11    2005     11    albers          90.288795              73.907   
12    2005     12    albers          90.245395              73.907   
13    2005     13   

In [12]:
# ============================================================
# MERGE INN I MASTER
# ============================================================

super_df = master.copy()

print(f"Master shape: {super_df.shape}")

# Merge historiske pit features
super_df = super_df.merge(
    pit_historical,
    on=["season", "round", "driver_id"],
    how="left"
)
print(f"Etter pit merge: {super_df.shape}")

# Merge historiske lap features
super_df = super_df.merge(
    lap_historical,
    on=["season", "round", "driver_id"],
    how="left"
)
print(f"Etter lap merge: {super_df.shape}")

# Sjekk at rader ikke endret seg
assert super_df.shape[0] == master.shape[0], "FEIL: Antall rader endret seg!"
print("✓ Antall rader er uendret")

Master shape: (25939, 46)
Etter pit merge: (25939, 49)
Etter lap merge: (25939, 53)
✓ Antall rader er uendret


In [13]:
# ============================================================
# VALIDER
# ============================================================

nye_kolonner = [
    "hist_avg_pit_stops",
    "hist_avg_pit_duration",
    "hist_best_pit_duration",
    "hist_avg_lap_time",
    "hist_best_lap_time",
    "hist_lap_consistency",
    "hist_best_lap_at_circuit"
]

print("NaN i nye kolonner:")
print(super_df[nye_kolonner].isnull().sum())

# Spot-sjekk - hamilton på 2012 løp bør ha 2011-data
hamilton_check = super_df[
    (super_df["driver_id"] == "hamilton") &
    (super_df["season"] == 2012)
][["season", "round", "driver_id"] + nye_kolonner].head(5)

print("\nSjekk Hamilton 2012 - bør ha historiske verdier:")
print(hamilton_check)

# Sjekk at første sesong (2011) ikke har historiske verdier
hamilton_2011 = super_df[
    (super_df["driver_id"] == "hamilton") &
    (super_df["season"] == 2011)
][["season", "round", "driver_id"] + nye_kolonner].head(3)

print("\nSjekk Hamilton 2011 - bør være NaN (første sesong med data):")
print(hamilton_2011)

NaN i nye kolonner:
hist_avg_pit_stops          20481
hist_avg_pit_duration       20481
hist_best_pit_duration      20481
hist_avg_lap_time           15042
hist_best_lap_time          15042
hist_lap_consistency        15047
hist_best_lap_at_circuit    17722
dtype: int64

Sjekk Hamilton 2012 - bør ha historiske verdier:
       season  round driver_id  hist_avg_pit_stops  hist_avg_pit_duration  \
19874    2012      1  hamilton            3.111111              22.423666   
19896    2012      2  hamilton            3.052632              22.462578   
19920    2012      3  hamilton            3.050000              22.648949   
19949    2012      4  hamilton            3.047619              22.550920   
19973    2012      5  hamilton            3.045455              22.778287   

       hist_best_pit_duration  hist_avg_lap_time  hist_best_lap_time  \
19874                  13.173          96.531560              72.506   
19896                  13.173          96.542178              72.506   


In [14]:
# ============================================================
# HÅNDTER NaN
# Mange NaN er forventet - eldre sesonger har ingen lap/pit data
# ============================================================

# Lag missing indicators FØR vi fyller NaN
super_df["missing_pit_history"] = (
    super_df["hist_avg_pit_stops"].isna()
).astype(int)

super_df["missing_lap_history"] = (
    super_df["hist_avg_lap_time"].isna()
).astype(int)

# Fyll NaN med 0
for col in nye_kolonner:
    super_df[col] = super_df[col].fillna(0)

print("NaN etter fylling:")
print(super_df[nye_kolonner].isnull().sum())

NaN etter fylling:
hist_avg_pit_stops          0
hist_avg_pit_duration       0
hist_best_pit_duration      0
hist_avg_lap_time           0
hist_best_lap_time          0
hist_lap_consistency        0
hist_best_lap_at_circuit    0
dtype: int64


In [15]:
# ============================================================
# LAGRE
# ============================================================

super_df.to_csv(
    "../data/processed/super_ml_dataset.csv",
    index=False
)

print(f"✓ Superdatasett lagret!")
print(f"Shape: {super_df.shape}")
print(f"\nNye kolonner lagt til:")
alle_nye = nye_kolonner + ["missing_pit_history", "missing_lap_history"]
for col in alle_nye:
    print(f"  - {col}")

✓ Superdatasett lagret!
Shape: (25939, 55)

Nye kolonner lagt til:
  - hist_avg_pit_stops
  - hist_avg_pit_duration
  - hist_best_pit_duration
  - hist_avg_lap_time
  - hist_best_lap_time
  - hist_lap_consistency
  - hist_best_lap_at_circuit
  - missing_pit_history
  - missing_lap_history
